# 01 — Reproduce LeWM (released checkpoints, official protocol)

**Purpose.**
- **Part A.** Evaluate the released LeWM checkpoints with the official config protocol
  (`third_party/le-wm/eval.py`, unmodified): 50 episodes × 3 evaluation seeds per environment.
  Writes one results JSON per env × seed and a reproduction table.
- **Part B.** A short *throughput probe* of the official training script (a few hundred
  optimisation steps on TwoRoom) so that the cost of one from-scratch run is measured
  before any long run is proposed. This is not a training run and produces no usable model.

**Expected runtime.** ~20 min (≈8 min dataset downloads, ≈6 min evaluation, ≈5 min probe).
Extrapolated from notebook 00 (≈2 s per evaluation episode); the probe time is unmeasured.

**Expected compute.** ≤ 0.5 L4-hours.

**How to run.** L4 runtime, set `REPO_URL`, Run all. Idempotent: finished evaluations are
skipped unless `FORCE = True`.

## 1. Config (the only cell you edit)

In [ ]:
# --- repository -------------------------------------------------------------
REPO_URL = "https://github.com/CHANGE_ME/jepa-memory.git"  # <-- your remote
REPO_REF = "main"
REPO_DIR = "/content/jepa-memory"

# --- Part A: reproduction ---------------------------------------------------
ENVS = ["tworoom", "pusht"]   # reacher / cube need INSTALL_EXTRA="wm-all,dev" (untested) + 24 / 46 GB
NUM_EVAL = None               # None -> protocol value (50)
EVAL_SEEDS = None             # None -> protocol value ([42, 43, 44])
INSTALL_EXTRA = "wm,dev"
FORCE = False

# --- Part B: training throughput probe --------------------------------------
RUN_TIMING_PROBE = True
PROBE_DATA = "tworoom"        # le-wm data config
PROBE_STEPS = 300             # optimisation steps at the official batch size (128)
PROBE_EPOCH_OPTIONS = (100, 10)  # official config value, paper-appendix value

# --- bookkeeping ------------------------------------------------------------
RUN_ID = "nb01-reproduce"
NOTEBOOK = "01_reproduce_lewm"
EST_HOURS = 0.5
SEED = 0                      # notebook-level seed (evaluation seeds are EVAL_SEEDS)
EXPECTED_GPUS = ("L4", "A100")

## 2. GPU check

In [ ]:
import subprocess

try:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu_name = ""
print("GPU:", gpu_name or "NONE")
HAS_GPU = bool(gpu_name)
if not any(g in gpu_name for g in EXPECTED_GPUS):
    print(f"WARNING: expected one of {EXPECTED_GPUS}. The official eval script needs CUDA "
          "(le-wm/eval.py:89); budget accounting assumes L4 or A100.")

## 3. Drive mount

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not on Colab: using repo-relative paths.")

## 4. Clone / update the repo (with the pinned `le-wm` submodule)

In [ ]:
def _git(*args, cwd=None):
    subprocess.run(["git", *args], cwd=cwd, check=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    assert "CHANGE_ME" not in REPO_URL, "Set REPO_URL in the config cell."
    clone_url = REPO_URL
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")
            if token:
                clone_url = REPO_URL.replace("https://", f"https://{token}@")
        except Exception:
            pass  # no secret configured -> public clone
    subprocess.run(["git", "clone", "-q", clone_url, REPO_DIR], check=True, capture_output=True)
    _git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)  # never keep the token on disk
else:
    _git("fetch", "-q", "--all", cwd=REPO_DIR)
_git("checkout", "-q", REPO_REF, cwd=REPO_DIR)
on_branch = subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO_DIR,
                           capture_output=True).returncode == 0
if on_branch:
    subprocess.run(["git", "pull", "-q", "--ff-only"], cwd=REPO_DIR)  # ok to fail offline
_git("submodule", "update", "--init", "--recursive", cwd=REPO_DIR)
os.chdir(REPO_DIR)
print("repo at", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["git", "submodule", "status"], capture_output=True, text=True).stdout.strip())

## 5. Install pinned dependencies and the `jmem` package

In [ ]:
# torch is NOT reinstalled: Colab's CUDA build satisfies torch>=2.4,<3 (see pyproject.toml).
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f".[{INSTALL_EXTRA}]"])
assert r.returncode == 0, "pip install failed (scroll up for the error)"
src = os.path.join(REPO_DIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)  # editable installs are only picked up by new processes
import jmem
print("jmem", jmem.__version__)

## 6. Paths, seeds, ledger

In [ ]:
import json, shutil
from jmem import paths, ledger, utils, datasets, aggregate
from jmem.evaluate import (run_official_eval, check_protocol, load_protocol,
                           official_record_to_result, write_result)
from jmem.train import run_official_train, estimate_run_hours, clips_per_epoch

assert HAS_GPU, "This notebook needs a CUDA runtime (le-wm/eval.py:89, train config accelerator: gpu)."
P = paths.setup()
utils.seed_everything(SEED)
ledger.start(RUN_ID, est_hours=EST_HOURS, notebook=NOTEBOOK)

protocol = load_protocol()
NUM_EVAL = NUM_EVAL or protocol["common"]["num_eval"]
EVAL_SEEDS = EVAL_SEEDS or protocol["eval_seeds"]
references = utils.load_yaml(paths.repo_root() / "configs" / "eval" / "references.yaml")
OUT = P.results / "reproduction"                     # Drive
REPO_OUT = paths.repo_root() / "results" / "reproduction"  # repo working tree
RAW = OUT / "raw"
for d in (OUT, REPO_OUT, RAW):
    d.mkdir(parents=True, exist_ok=True)
utils.save_resolved_config(
    {"envs": ENVS, "num_eval": NUM_EVAL, "eval_seeds": EVAL_SEEDS, "protocol": protocol,
     "references": references, "probe": {"run": RUN_TIMING_PROBE, "data": PROBE_DATA, "steps": PROBE_STEPS}},
    OUT, name="nb01")
print("envs", ENVS, "| episodes", NUM_EVAL, "| eval seeds", EVAL_SEEDS, "| cpu cores", os.cpu_count())

## 7. Released checkpoints and datasets

In [ ]:
for env in sorted(set(ENVS) | ({PROBE_DATA} if RUN_TIMING_PROBE else set())):
    datasets.ensure_checkpoint(env)
    datasets.ensure_dataset(env, P.data_home)

## 8. Part A — official evaluation, 50 episodes × 3 seeds

`seed` selects the evaluation episodes *and* seeds the CEM noise (`le-wm/eval.py:123`,
`config/eval/solver/cem.yaml:9`), so the three seeds are three different episode sets.

In [ ]:
RESULTS, BAD = [], {}
for env in ENVS:
    for seed in EVAL_SEEDS:
        rec = run_official_eval(env, datasets.CHECKPOINTS[env], out_dir=RAW,
                                num_eval=NUM_EVAL, seed=seed, paths=P, force=FORCE)
        res = official_record_to_result(rec, protocol)
        path = write_result(res, OUT)
        shutil.copy2(path, REPO_OUT / path.name)
        RESULTS.append(res)
        if res["protocol"]["mismatches"]:
            BAD[res["run_id"]] = res["protocol"]["mismatches"]
        off = str(protocol["common"]["goal_offset_steps"])
        print(f"{res['run_id']}: success@{off} = {100 * res['success'][off]:.1f}% "
              f"({res['n_episodes']} episodes, {rec['evaluate_wallclock_s']:.0f} s eval)")

## 9. Reproduction table (generated from the results JSON files)

In [ ]:
rows = aggregate.summarize(aggregate.load_results(OUT, pattern="official-*.json"))
table = aggregate.reproduction_table(rows, references["references"], references["tolerance"])
(OUT / "reproduction_table.md").write_text(table + "\n")
shutil.copy2(OUT / "reproduction_table.md", REPO_OUT / "reproduction_table.md")
print(table)

## 10. Part B — training throughput probe (official `train.py`, a few hundred steps)

Runs `third_party/le-wm/train.py` unchanged with `+trainer.max_steps=PROBE_STEPS` and no
validation, and times every optimisation step. The extrapolation excludes validation,
per-epoch checkpoint writes and dataloader restarts, so it is a **lower bound**.

In [ ]:
PROBE = None
_probe_json = OUT / "train_timing_probe.json"
if RUN_TIMING_PROBE and _probe_json.exists() and not FORCE:
    PROBE = json.loads(_probe_json.read_text())
    print("[probe] reusing", _probe_json)
elif RUN_TIMING_PROBE:
    PROBE = run_official_train(
        PROBE_DATA, "timing_probe", OUT, paths=P,
        extra_overrides=[f"+trainer.max_steps={PROBE_STEPS}", "+trainer.limit_val_batches=0",
                         "trainer.max_epochs=1"])
    import h5py
    with h5py.File(datasets.dataset_path(PROBE_DATA, P.data_home), "r") as f:
        ep_len = f["ep_len"][:]
    base = utils.load_yaml(paths.repo_root() / "configs" / "base.yaml")
    spe = clips_per_epoch(ep_len, base["data"]["num_steps"], base["data"]["frameskip"],
                          base["train"]["train_split"], base["train"]["batch_size"])
    PROBE["estimates"] = {str(e): estimate_run_hours(PROBE, e, spe) for e in PROBE_EPOCH_OPTIONS}
    (OUT / "train_timing_probe.json").write_text(json.dumps(PROBE, indent=2))
    shutil.copy2(OUT / "train_timing_probe.json", REPO_OUT / "train_timing_probe.json")
if PROBE is not None:
    print(json.dumps({k: PROBE[k] for k in ("step_time", "batch_size", "num_workers", "precision",
                                            "device", "global_step", "estimates")}, indent=1))

## 11. Validation

In [ ]:
import math

assert len(RESULTS) == len(ENVS) * len(EVAL_SEEDS)
off = str(protocol["common"]["goal_offset_steps"])
for res in RESULTS:
    assert (OUT / f"{res['run_id']}.json").exists() and (REPO_OUT / f"{res['run_id']}.json").exists()
    v = res["success"][off]
    assert not math.isnan(v) and 0.0 <= v <= 1.0, v
    assert res["n_episodes"] == NUM_EVAL
assert not BAD, f"runs deviated from configs/eval/protocol.yaml: {BAD}"
assert len(aggregate.load_results(REPO_OUT, pattern="official-*.json")) >= len(RESULTS)  # all validate
if RUN_TIMING_PROBE:
    assert PROBE["finished"] and PROBE["global_step"] == PROBE_STEPS, PROBE["global_step"]
    assert PROBE["step_time"]["median_s"] > 0
print("all validation checks passed")

## 12. Ledger and summary (paste everything below back to Claude)

In [ ]:
row = ledger.stop()
budget = ledger.summary()
archive = shutil.make_archive(str(P.results / "reproduction_results"), "zip", REPO_OUT)

print("\n===== NB01 SUMMARY (paste back) =====")
print("repo_sha   :", utils.git_sha())
print("gpu        :", gpu_name, "| cpu cores:", os.cpu_count())
print("ledger_row :", json.dumps(row))
print("ledger_used_L4h / projected:", round(budget.used, 3), "/", round(budget.projected, 3))
print("results zip:", archive, "(unzip into the repo's results/reproduction/ and commit)")
print()
print(table)
print()
for res in RESULTS:
    eps = res["episode_successes"][off]
    print(res["run_id"], "| k/n =", sum(eps), "/", len(eps), "| bits =", "".join("1" if e else "0" for e in eps),
          "| mismatches =", res["protocol"]["mismatches"])
if RUN_TIMING_PROBE:
    print()
    print("probe      :", json.dumps({k: PROBE[k] for k in ("step_time", "batch_size", "num_workers",
          "precision", "device", "global_step", "steps_per_epoch", "fit_wallclock_s", "wallclock_s")}))
    for e, est in PROBE["estimates"].items():
        print(f"estimate   : {e} epochs -> {est['hours_total']:.1f} {PROBE['hardware']}-hours "
              f"(lower bound; {est['steps_per_epoch']:.0f} steps/epoch, {est['hours_per_epoch']:.2f} h/epoch)")